# 🎓 Student Performance Prediction
This notebook demonstrates a machine learning pipeline with SQL integration to predict student academic performance.

In [ ]:
# Step 1: Import Libraries
import pandas as pd
import numpy as np
import sqlite3
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# Step 2: Load Dataset
df = pd.read_csv('data/student-mat.csv', sep=';')

In [ ]:
# Step 3: Store dataset into SQLite
conn = sqlite3.connect('students.db')
df.to_sql('student_data', conn, if_exists='replace', index=False)

In [ ]:
# Step 4: SQL Queries for EDA
print(pd.read_sql_query("SELECT sex, AVG(G3) AS avg_final_grade FROM student_data GROUP BY sex;", conn))

In [ ]:
# Step 5: Create SQL View for ML-ready data
conn.execute("""
CREATE VIEW IF NOT EXISTS ml_ready_data AS
SELECT sex, age, address, famsize, Pstatus, Medu, Fedu, studytime,
       failures, absences, G1, G2, G3
FROM student_data
WHERE G3 IS NOT NULL;
""")

df = pd.read_sql_query("SELECT * FROM ml_ready_data", conn)

In [ ]:
# Step 6: Encode categorical variables
label_encoders = {}
for col in df.select_dtypes(include='object').columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

In [ ]:
# Step 7: Define Target variable
df['performance'] = pd.cut(df['G3'], bins=[-1, 10, 15, 20], labels=['Low', 'Medium', 'High'])
df.drop('G3', axis=1, inplace=True)

In [ ]:
# Step 8: Train-Test Split
X = df.drop('performance', axis=1)
y = df['performance']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [ ]:
# Step 9: Feature Scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
# Step 10: Train RandomForest Model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

In [ ]:
# Step 11: Evaluate Model
y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

In [ ]:
# Step 12: Feature Importance
importances = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
plt.figure(figsize=(10, 6))
sns.barplot(x=importances, y=importances.index)
plt.title("Feature Importance from Random Forest")
plt.xlabel("Importance Score")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()